In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
from CellType_PSY import *
#import scanpy as sc
HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

try:
    os.chdir(f"{ProjDIR}/notebooks/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
Bias_Save_Dir = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/dat/Spec_Bias_Jul07/"
Bias_Null_Dir = Bias_Save_Dir + "CTRL/RandGeneBias_Spec/"

### ASD/SCZ

In [ ]:
RandG_SCZ_top61_DIR = "{}/RandGene_SCZ.top61".format(Bias_Null_Dir)
RandG_SCZ_top61_DFs = LoadNullDF(RandG_SCZ_top61_DIR)

SCZ_Bias_top61 = pd.read_csv("{}/HCT.SCZ61.csv".format(Bias_Save_Dir), index_col=0)
SCZ_Bias_top61_P_randG = AddPvalue_optimized(SCZ_Bias_top61, RandG_SCZ_top61_DFs)
SCZ_Bias_top61_P_randG.to_csv("{}/HCT.SCZ61.addP.csv".format(Bias_Save_Dir))
SuperClusterBias_BoxPlot(SCZ_Bias_top61_P_randG, "SCZ", NeuroOnly=False, sortby="mean", EffectCol="-logP")

In [ ]:
VIP_Anno = pd.read_csv("VIP_Anno.csv", index_col=0)
common_indices = SCZ_Bias_top61_P_randG.index.intersection(VIP_Anno.index)
SCZ_Bias_top61_P_randG_sub = SCZ_Bias_top61_P_randG.loc[common_indices].copy()
SCZ_Bias_top61_P_randG_sub["VIP"] = VIP_Anno.loc[common_indices, "VIP"]

In [ ]:
SCZ_Bias_top61_P_randG_sub.head(2)

In [ ]:
VIP_pos = SCZ_Bias_top61_P_randG_sub[SCZ_Bias_top61_P_randG_sub["VIP"] >= 1]
VIP_neg = SCZ_Bias_top61_P_randG_sub[SCZ_Bias_top61_P_randG_sub["VIP"] < 1]
# plot effect of VIP+ vs VIP-
EFFECT = "-logP"
data = [VIP_pos[EFFECT], VIP_neg[EFFECT]]
# Perform Mann-Whitney U test
stat, pval = scipy.stats.mannwhitneyu(VIP_pos[EFFECT], 
                                    VIP_neg[EFFECT])
# Create boxplot with individual points
bp = plt.boxplot(data, labels=["VIP+", "VIP-"])
# Add scatter points
for i, d in enumerate([VIP_pos[EFFECT], VIP_neg[EFFECT]]):
    x = np.random.normal(i+1, 0.04, size=len(d))
    plt.scatter(x, d, alpha=0.4, s=20)
plt.ylabel("Effect")
plt.title(f"p = {pval:.2e}")
plt.show()


In [ ]:
N_VIP_Sig = VIP_pos[VIP_pos["q-value"]< 0.05].shape[0]
N_nonVIP_Sig = VIP_neg[VIP_neg["q-value"]< 0.05].shape[0]
N_total_VIP = VIP_pos.shape[0]
N_total_nonVIP = VIP_neg.shape[0]
print(f"VIP+: {N_VIP_Sig}, VIP-: {N_nonVIP_Sig}")
print(f"VIP+: {N_VIP_Sig/N_total_VIP}, VIP-: {N_nonVIP_Sig/N_total_nonVIP}")

In [ ]:
# Test if N_VIP_Sig is significantly more than expected by chance using a binomial test
from scipy.stats import binom_test

# Assume the expected proportion of significant genes is the same as in VIP- group
expected_prop = N_nonVIP_Sig / N_total_nonVIP if N_total_nonVIP > 0 else 0

# Perform one-sided binomial test: is N_VIP_Sig greater than expected under null?
p_binom = binom_test(N_VIP_Sig, N_total_VIP, expected_prop, alternative='greater')

print(f"Binomial test p-value (VIP+ > expected): {p_binom:.3g}")
N_VIP_Sig, N_nonVIP_Sig, N_total_VIP, N_total_nonVIP

In [ ]:
SuperCluster_SCZ_df = test_all_superclusters_vectorized(ALL_CTs, Anno, SCZ_Bias_top61_P_randG, RandG_SCZ_top61_DFs)
SuperCluster_SCZ_df.to_csv("{}/SuperCluster_SCZ.Pvalue.csv".format(Bias_Save_Dir))
SuperCluster_SCZ_df

In [ ]:
RandG_ASD_HIQ_top61_DIR = "{}/RandGene_ASD_HIQ.top61/".format(Bias_Null_Dir)
RandG_ASD_HIQ_top61_DFs = LoadNullDF(RandG_ASD_HIQ_top61_DIR)
RandG_ASD_LIQ_top61_DIR = "{}/RandGene_ASD_LIQ.top61/".format(Bias_Null_Dir)
RandG_ASD_LIQ_top61_DFs = LoadNullDF(RandG_ASD_LIQ_top61_DIR)

In [ ]:
ASD_HIQ_Bias_top61 = pd.read_csv("{}/HCT.ASD61.HIQ.csv".format(Bias_Save_Dir), index_col=0)
ASD_HIQ_Bias_top61_P_randG = AddPvalue_optimized(ASD_HIQ_Bias_top61, RandG_ASD_HIQ_top61_DFs)
ASD_LIQ_Bias_top61 = pd.read_csv("{}/HCT.ASD61.LIQ.csv".format(Bias_Save_Dir), index_col=0)
ASD_LIQ_Bias_top61_P_randG = AddPvalue_optimized(ASD_LIQ_Bias_top61, RandG_ASD_LIQ_top61_DFs)

ASD_HIQ_Bias_top61_P_randG.to_csv("{}/HCT.ASD61.HIQ.P_randG.csv".format(Bias_Save_Dir), index=False)
ASD_LIQ_Bias_top61_P_randG.to_csv("{}/HCT.ASD61.LIQ.P_randG.csv".format(Bias_Save_Dir), index=False)

In [ ]:
ASD_HIQ_Bias_top61_P_randG["-logP"] = -np.log10(ASD_HIQ_Bias_top61_P_randG["P-value"])
SuperClusterBias_BoxPlot(ASD_HIQ_Bias_top61_P_randG, "ASD_HIQ", NeuroOnly=False, sortby="mean", EffectCol="-logP")
ASD_LIQ_Bias_top61_P_randG["-logP"] = -np.log10(ASD_LIQ_Bias_top61_P_randG["P-value"])
SuperClusterBias_BoxPlot(ASD_LIQ_Bias_top61_P_randG, "ASD_LIQ", NeuroOnly=False, sortby="mean", EffectCol="-logP")

In [ ]:
SuperCluster_ASD_HIQ_df = test_all_superclusters_vectorized(ALL_CTs, Anno, ASD_HIQ_Bias_top61_P_randG, RandG_ASD_HIQ_top61_DFs)
SuperCluster_ASD_LIQ_df = test_all_superclusters(ALL_CTs, Anno, ASD_LIQ_Bias_top61_P_randG, RandG_ASD_LIQ_top61_DFs)
#SuperCluster_ASD_HIQ_df

In [ ]:
SuperCluster_ASD_HIQ_df

In [ ]:
SuperCluster_ASD_LIQ_df

In [ ]:
#RandG_DDD_DIR = "{}/RandGene_DDD.top61_Jun11/".format(Bias_Null_Dir)
RandG_DDD_top61_DIR = "{}/RandGene_DDD.top61/".format(Bias_Null_Dir)
RandG_DDD_top61_DFs = LoadNullDF(RandG_DDD_top61_DIR)

In [ ]:
DDD_Bias = pd.read_csv("{}/HCT.DDDHC.Spec.top61.csv".format(Bias_Save_Dir), index_col=0)
DDD_Bias_P_randG = AddPvalue_optimized(DDD_Bias, RandG_DDD_top61_DFs)
DDD_Bias_P_randG.to_csv("{}/HCT.DDD.top61.addP.csv".format(Bias_Save_Dir))
#DDD_Bias_P_randG.head(20)
SuperClusterBias_BoxPlot(DDD_Bias_P_randG, "DDD", NeuroOnly=False, sortby="mean", EffectCol="-logP")

In [ ]:
SuperCluster_DDD_df = test_all_superclusters(ALL_CTs, Anno, DDD_Bias_P_randG, RandG_DDD_top61_DFs)
SuperCluster_DDD_df 

In [ ]:
RandG_22q_DIR = "{}/RandGene_22q_del/".format(Bias_Null_Dir)
RandG_22q_top61_DFs = LoadNullDF(RandG_22q_DIR)

In [ ]:
X22q_Bias = pd.read_csv("{}/HCT.X22q.csv".format(Bias_Save_Dir), index_col=0)
X22q_Bias_P_randG = AddPvalue_optimized(X22q_Bias, RandG_22q_top61_DFs)
X22q_Bias_P_randG.to_csv("{}/HCT.X22q.addP.csv".format(Bias_Save_Dir))
#X22q_Bias_P_randG.head(20)
SuperClusterBias_BoxPlot(X22q_Bias_P_randG, "X22q", NeuroOnly=False, sortby="mean", EffectCol="-logP")

In [ ]:
SuperCluster_22q_df = test_all_superclusters_vectorized(ALL_CTs, Anno, X22q_Bias_P_randG, RandG_22q_top61_DFs)
SuperCluster_22q_df

In [ ]:
PlotQQ([SCZ_Bias_top61_P_randG, ASD_HIQ_Bias_top61_P_randG, ASD_LIQ_Bias_top61_P_randG, X22q_Bias_P_randG, DDD_Bias_P_randG], 
[ "SCZ", "ASD_HIQ", "ASD_LIQ", "22q", "NDD"])

In [ ]:
RandG_VNR_DIR = "{}/RandGene_UKBB_VNR.top61/".format(Bias_Null_Dir)
RandG_VNR_top61_DFs = LoadNullDF(RandG_VNR_DIR)
RandG_EDU_DIR = "{}/RandGene_UKBB_EDU.top61/".format(Bias_Null_Dir)
RandG_EDU_top61_DFs = LoadNullDF(RandG_EDU_DIR)

In [ ]:
UKBB_VNR_Neg_Bias = pd.read_csv("{}/HCT.VNR.Neg.top61.csv".format(Bias_Save_Dir), index_col=0)
UKBB_VNR_Neg_Bias_P_randG = AddPvalue_optimized(UKBB_VNR_Neg_Bias, RandG_VNR_top61_DFs)

UKBB_EDU_Neg_Bias = pd.read_csv("{}/HCT.EDU.Neg.top61.csv".format(Bias_Save_Dir), index_col=0)
UKBB_EDU_Neg_Bias_P_randG = AddPvalue_optimized(UKBB_EDU_Neg_Bias, RandG_EDU_top61_DFs)


In [ ]:
UKBB_VNR_Pos_Bias = pd.read_csv("{}/HCT.VNR.Pos.top61.csv".format(Bias_Save_Dir), index_col=0)
UKBB_VNR_Pos_Bias_P_randG = AddPvalue_optimized(UKBB_VNR_Pos_Bias, RandG_VNR_top61_DFs)

UKBB_EDU_Pos_Bias = pd.read_csv("{}/HCT.EDU.Pos.top61.csv".format(Bias_Save_Dir), index_col=0)
UKBB_EDU_Pos_Bias_P_randG = AddPvalue_optimized(UKBB_EDU_Pos_Bias, RandG_EDU_top61_DFs)

In [ ]:
UKBB_VNR_Neg_Bias_P_randG.to_csv(Bias_Save_Dir + "UKBB_VNR_Neg_Bias_addP.csv")
UKBB_EDU_Neg_Bias_P_randG.to_csv(Bias_Save_Dir + "UKBB_EDU_Neg_Bias_addP.csv")

UKBB_VNR_Pos_Bias_P_randG.to_csv(Bias_Save_Dir + "UKBB_VNR_Pos_Bias_addP.csv")
UKBB_EDU_Pos_Bias_P_randG.to_csv(Bias_Save_Dir + "UKBB_EDU_Pos_Bias_addP.csv")

In [ ]:
SuperClusterBias_BoxPlot(UKBB_VNR_Pos_Bias_P_randG, "VNR+", NeuroOnly=False, sortby="mean", EffectCol="-logP")
SuperClusterBias_BoxPlot(UKBB_EDU_Pos_Bias_P_randG, "EDU+", NeuroOnly=False, sortby="mean", EffectCol="-logP")

In [ ]:
SuperClusterBias_BoxPlot(UKBB_VNR_Neg_Bias_P_randG, "VNR-", NeuroOnly=False, sortby="mean", EffectCol="-logP")
SuperClusterBias_BoxPlot(UKBB_EDU_Neg_Bias_P_randG, "EDU-", NeuroOnly=False, sortby="mean", EffectCol="-logP")